In [8]:
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

# Student Identity Inputs
LAST_NAME = "De Jesus"
STUDENT_ID = "TUPM-26-1944"
SEED_DIGIT = int(STUDENT_ID[-1])

# Data Generation Rules
TARGET_YEAR = 2015 + SEED_DIGIT

# Find the project folder so paths work from the notebook or a subfolder.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data").exists() and (candidate / "output").exists()
    ),
    Path.cwd(),
)
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROJECT_ROOT / "Global_Cybersecurity_Threats_2015-2024.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = PROJECT_ROOT / "data" / "Global_Cybersecurity_Threats_2015-2024.csv"

FILTERED_CSV_PATH = OUTPUT_DIR / f"filtered_cybersecurity_{TARGET_YEAR}.csv"
SUMMARY_JSON_PATH = OUTPUT_DIR / f"cybersecurity_summary_{TARGET_YEAR}.json"
BACKUP_JSON_PATH = OUTPUT_DIR / f"cybersecurity_summary_{TARGET_YEAR}_backup.json"
REPORT_TXT_PATH = OUTPUT_DIR / f"cybersecurity_report_{TARGET_YEAR}.txt"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Cybersecurity dataset not found: {DATASET_PATH}. "
        "Place Global_Cybersecurity_Threats_2015-2024.csv in the project root or data folder."
    )

# Data Containers
total_dataset_records = 0
target_year_incidents = []
attack_type_counts = Counter()
total_financial_loss = 0.0

# Read the dataset and collect incidents from the target year.
with DATASET_PATH.open(mode="r", encoding="utf-8-sig", errors="ignore", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_dataset_records += 1

        year_value = row.get("Year") or row.get("Timestamp") or row.get("Date") or ""
        try:
            year = int(str(year_value).strip()[:4])
        except ValueError:
            continue

        if year != TARGET_YEAR:
            continue

        attack_type = row.get("Attack Type") or "Unknown"
        loss_value = row.get("Financial Loss (in Million $)") or "0"
        try:
            loss = float(str(loss_value).replace("$", "").replace(",", "").strip())
        except ValueError:
            loss = 0.0

        incident = {
            "Timestamp": row.get("Timestamp") or row.get("Date") or str(year),
            "Attack Type": attack_type,
            "Financial Loss (in Million $)": loss,
        }
        if row.get("Incident ID"):
            incident["Incident ID"] = row["Incident ID"]

        target_year_incidents.append(incident)
        attack_type_counts[attack_type] += 1
        total_financial_loss += loss

incidents_count = len(target_year_incidents)
avg_financial_loss = (
    total_financial_loss / incidents_count if incidents_count > 0 else 0.0
)

top_5_costliest = sorted(
    target_year_incidents,
    key=lambda incident: incident["Financial Loss (in Million $)"],
    reverse=True,
)[:5]

# Export filtered incidents to CSV.
fieldnames = ["Timestamp", "Attack Type", "Financial Loss (in Million $)"]
with FILTERED_CSV_PATH.open(mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(
        {key: incident.get(key, "") for key in fieldnames}
        for incident in target_year_incidents
    )

# Export the JSON summary and verify its backup.
summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": total_dataset_records,
    "incidents_in_target_year": incidents_count,
    "count_per_attack_type": dict(attack_type_counts),
    "total_financial_loss_usd_millions": round(total_financial_loss, 2),
    "average_financial_loss_usd_millions": round(avg_financial_loss, 2),
    "data_source_note": "Synthetic demo data; not real incident records.",
}

with SUMMARY_JSON_PATH.open(mode="w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

shutil.copy2(SUMMARY_JSON_PATH, BACKUP_JSON_PATH)
backup_verified = (
    BACKUP_JSON_PATH.exists()
    and BACKUP_JSON_PATH.read_bytes() == SUMMARY_JSON_PATH.read_bytes()
)

report_lines = [
    "=" * 70,
    f"{'EXERCISE 3: CYBERSECURITY INCIDENT REPORT':^70}",
    "=" * 70,
    f"Target Year            : {TARGET_YEAR}",
    "Data Source            : Synthetic demo data; not real incidents",
    f"Total Dataset Records  : {total_dataset_records}",
    f"Incidents in {TARGET_YEAR}     : {incidents_count}",
    f"Attack Type Counts     : {dict(attack_type_counts)}",
    f"Total Financial Loss   : ${total_financial_loss:,.2f} Million",
    f"Avg Financial Loss     : ${avg_financial_loss:,.2f} Million",
    "-" * 70,
    "Five Costliest Incidents:",
]
for idx, incident in enumerate(top_5_costliest, start=1):
    incident_id = incident.get("Incident ID", "N/A")
    loss = incident["Financial Loss (in Million $)"]
    attack_type = incident["Attack Type"]
    report_lines.append(
        f"  {idx}. ID: {incident_id} | Loss: ${loss:,.2f}M | Type: {attack_type}"
    )
report_lines.extend([
    "-" * 70,
    f"Filtered CSV File      : {FILTERED_CSV_PATH.name}",
    f"JSON Summary File      : {SUMMARY_JSON_PATH.name}",
    f"Backup Verification    : {'VERIFIED' if backup_verified else 'FAILED'}",
    f"Text Report File       : {REPORT_TXT_PATH.name}",
    "=" * 70,
])

report_text = "\n".join(report_lines)
REPORT_TXT_PATH.write_text(report_text + "\n", encoding="utf-8")
print(report_text)


              EXERCISE 3: CYBERSECURITY INCIDENT REPORT               
Target Year            : 2019
Data Source            : Synthetic demo data; not real incidents
Total Dataset Records  : 30
Incidents in 2019     : 3
Attack Type Counts     : {'Ransomware': 1, 'Phishing': 1, 'Data Breach': 1}
Total Financial Loss   : $16.45 Million
Avg Financial Loss     : $5.48 Million
----------------------------------------------------------------------
Five Costliest Incidents:
  1. ID: CYB-2019-001 | Loss: $7.80M | Type: Ransomware
  2. ID: CYB-2019-003 | Loss: $6.40M | Type: Data Breach
  3. ID: CYB-2019-002 | Loss: $2.25M | Type: Phishing
----------------------------------------------------------------------
Filtered CSV File      : filtered_cybersecurity_2019.csv
JSON Summary File      : cybersecurity_summary_2019.json
Backup Verification    : VERIFIED
Text Report File       : cybersecurity_report_2019.txt
